In [11]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

# Logistic Growth Model

The logistic ODE is defined as:$$\frac{dL(t)}{dt} = a L(t) - b L(t)^2$$Here, $L(t)$ represents the population or labor size, $a$ is the intrinsic growth rate, and $b$ represents the competition or carrying capacity parameter (Jo et al., 2024).

### Parameter Space

<ul>
  <li>Intrinsic growth rate ($a$): $[0.1, 0.6]$</li>
  <li>Competition parameter ($b$): $[0.001, 0.01]$</li>
  <li>Initial population size ($L(0)$): $[5.0, 50.0]$</li>
</ul>

In [13]:
def logistic_ode(t, L, a, b):
    """
    Defines the continuous Logistic Growth ODE.
    dL/dt = a * L - b * L^2
    """
    dL_dt = a * L - b * (L ** 2)
    return dL_dt

def generate_logistic_dataset(num_samples=500, T=50, num_steps=100):
    """
    Generates training points for the Logistic Growth Model.
    
    Args:
        num_samples (int): Number of different parameter combinations (M).
        T (float): Time horizon.
        num_steps (int): Number of time points to evaluate.
        
    Returns:
        X_inputs (np.ndarray): Shape (M, 3) - [L(0), a, b]
        Y_trajectories (np.ndarray): Shape (M, num_steps) - L(t)
    """
    t_span = (0.0, T)
    t_eval = np.linspace(0.0, T, num_steps)
    
    X_inputs = []
    Y_trajectories = []
    
    np.random.seed(42)
    
    for _ in range(num_samples):
        # 1. Sample parameters from defined uniform ranges
        a = np.random.uniform(0.1, 0.6)
        b = np.random.uniform(0.001, 0.01)
        L0 = np.random.uniform(5.0, 50.0)
        
        # 2. Solve the ODE numerically using scipy.integrate.solve_ivp
        sol = solve_ivp(
            logistic_ode, 
            t_span, 
            [L0], 
            args=(a, b), 
            t_eval=t_eval,
            method='RK45'
        )
        
        # 3. Append inputs and outputs
        X_inputs.append([L0, a, b])
        Y_trajectories.append(sol.y[0])
        
    return np.array(X_inputs), np.array(Y_trajectories)

In [14]:
X_inputs, Y_trajectories = generate_logistic_dataset(num_samples=500)

input_labels = ['L0_initial', 'growth_rate_a', 'competition_coeff_b']

num_steps = Y_trajectories.shape[1]
trajectory_labels = [f'L_t{i}' for i in range(num_steps)]

full_data_matrix = np.concatenate([X_inputs, Y_trajectories], axis=1)

df_logistic = pd.DataFrame(full_data_matrix, columns=input_labels + trajectory_labels)

# Save to CSV
file_name = 'logistic_growth_dataset.csv'
df_logistic.to_csv(file_name, index=False)

# The Ramsey-Cass-Koopmans (RCK) Growth System

3. The Ramsey-Cass-Koopmans (RCK) Growth System The RCK model consists of a coupled system of two ODEs that jointly determine consumption and capital accumulation paths.$$\dot{k}(t) = f(k(t)) - c(t) - (n + g + \delta)k(t)$$$$\dot{c}(t) = \frac{c(t)}{\theta} [f'(k(t)) - \delta - \rho - \theta g]$$Variables: $k(t)$ is the capital per effective worker, $c(t)$ is the consumption per effective worker, $\rho$ is the discount rate, $\theta$ is the coefficient of relative risk aversion, and $g$ is the technological growth rate.

The RCK model is a system of two coupled ordinary differential equations that endogenizes the savings rate through representative household optimization (Wikipedia, 2026). The continuous dynamic system is defined as:  $$\dot{k}(t) = f(k(t)) - c(t) - (n + g + \delta)k(t)$$$$\dot{c}(t) = \frac{c(t)}{\theta} [f'(k(t)) - \delta - \rho - \theta g]$$Here, $k(t)$ is the capital per effective worker, $c(t)$ is the consumption per effective worker, $\alpha$ is the capital share of output, $\rho$ is the rate of time preference, $\theta$ is the inverse of the intertemporal elasticity of substitution, $\delta$ is the depreciation rate, $n$ is the population growth rate, and $g$ is the technological growth rate (Haase, 2025).1. Parameter and Initial State SpaceBecause the RCK model exhibits saddle-path stability, trajectories that do not converge will diverge or lead to negative values (MathWorks, 2026). To ensure stable numerical solutions, the initial states $k(0)$ and $c(0)$ are sampled within the neighborhood of the steady state $(k^*, c^*)$.  $$\text{Steady State Capital: } k^* = \left( \frac{\alpha}{\delta + \rho + \theta g} \right)^{\frac{1}{1-\alpha}}$$$$\text{Steady State Consumption: } c^* = (k^*)^{\alpha} - (n + g + \delta)k^*$$The bounds used for generating the random parameters are as follows:Capital elasticity ($\alpha$): $[0.3, 0.4]$Depreciation rate ($\delta$): $[0.03, 0.08]$Time preference rate ($\rho$): $[0.01, 0.05]$Risk aversion parameter ($\theta$): $[1.5, 3.0]$Population growth rate ($n$): $[0.01, 0.03]$Technological growth rate ($g$): $[0.01, 0.03]$Initial conditions $(k_0, c_0)$  $[0.8k^*, 1.2k^*]$ and $[0.8c^*, 1.2c^*]$.

In [16]:
import numpy as np
from scipy.integrate import solve_ivp

def rck_ode(t, z, alpha, delta, rho, theta, n, g):
    """
    Defines the coupled RCK ODE system.
    z[0] = k(t), z[1] = c(t)
    """
    k, c = z
    f_k = k ** alpha
    f_prime = alpha * (k ** (alpha - 1))
    
    dk_dt = f_k - c - (n + g + delta) * k
    dc_dt = (c / theta) * (f_prime - delta - rho - theta * g)
    
    return [dk_dt, dc_dt]

def generate_rck_dataset(num_samples=500, T=20, num_steps=100):
    """
    Generates training points for the RCK Growth Model.
    
    Args:
        num_samples (int): Number of different parameter combinations (M).
        T (float): Time horizon.
        num_steps (int): Number of time points to evaluate.
        
    Returns:
        X_inputs (np.ndarray): Shape (M, 8) - [alpha, delta, rho, theta, n, g, k(0), c(0)]
        Y_k (np.ndarray): Shape (M, num_steps) - k(t)
        Y_c (np.ndarray): Shape (M, num_steps) - c(t)
    """
    t_span = (0.0, T)
    t_eval = np.linspace(0.0, T, num_steps)
    
    X_inputs = []
    Y_k = []
    Y_c = []
    
    np.random.seed(42)
    
    while len(X_inputs) < num_samples:
        # Sample parameters
        alpha = np.random.uniform(0.3, 0.4)
        delta = np.random.uniform(0.03, 0.08)
        rho = np.random.uniform(0.01, 0.05)
        theta = np.random.uniform(1.5, 3.0)
        n = np.random.uniform(0.01, 0.03)
        g = np.random.uniform(0.01, 0.03)
        
        # Calculate steady state
        try:
            k_star = ((alpha) / (delta + rho + theta * g)) ** (1 / (1 - alpha))
            c_star = (k_star ** alpha) - (n + g + delta) * k_star
            
            if k_star <= 0 or c_star <= 0:
                continue
        except (ZeroDivisionError, ValueError):
            continue
            
        # Sample initial values near the steady state to avoid divergence
        k0 = np.random.uniform(0.8 * k_star, 1.2 * k_star)
        c0 = np.random.uniform(0.8 * c_star, 1.2 * c_star)
        
        sol = solve_ivp(
            rck_ode, 
            t_span, 
            [k0, c0], 
            args=(alpha, delta, rho, theta, n, g), 
            t_eval=t_eval,
            method='RK45'
        )
        
        if sol.success and sol.y.shape[1] == num_steps:
            # Check for non-divergent outputs
            if np.max(sol.y) < 1e4 and np.min(sol.y) > 0:
                X_inputs.append([alpha, delta, rho, theta, n, g, k0, c0])
                Y_k.append(sol.y[0])
                Y_c.append(sol.y[1])
                
    return np.array(X_inputs), np.array(Y_k), np.array(Y_c)



In [17]:

X_inputs, Y_k, Y_c = generate_rck_dataset(num_samples=500)
input_labels = [
    'alpha', 'delta', 'rho', 'theta', 'n', 'g', 'k0_initial', 'c0_initial'
]

num_steps = Y_k.shape[1]
k_labels = [f'k_t{i}' for i in range(num_steps)]
c_labels = [f'c_t{i}' for i in range(num_steps)]

full_data_matrix = np.hstack([X_inputs, Y_k, Y_c])


all_labels = input_labels + k_labels + c_labels
df_rck = pd.DataFrame(full_data_matrix, columns=all_labels)


file_name = 'rck_growth_dataset.csv'
df_rck.to_csv(file_name, index=False)



C:\Users\antek\AppData\Local\Temp\ipykernel_28132\1218264237.py:10: RuntimeWarning: invalid value encountered in scalar power
  f_k = k ** alpha
C:\Users\antek\AppData\Local\Temp\ipykernel_28132\1218264237.py:11: RuntimeWarning: invalid value encountered in scalar power
  f_prime = alpha * (k ** (alpha - 1))
